In [ ]:
diabetes <- read.csv("diabetes.csv")
# 0 = No Diabetes
# 1 = Diabetes

In [ ]:
# Training-Testing Split
set.seed(123) # Set Seed
# Train on 'diabetes' (want same results for class & global)
train_idx <- sample(
  1:nrow(diabetes),
  size = 0.8 * nrow(diabetes)
)

In [ ]:
# Create train/test sets from the original data
  # Original Dataset
train_original <- diabetes[train_idx, ]
test_original  <- diabetes[-train_idx, ]

# Logistic Regression

In [ ]:
# Logistic Regression
model_original <- glm(
  Outcome ~ .,
  data = train_original,
  family = binomial
)

prob_original <- predict(
  model_original,
  newdata = test_original,
  type = "response"
)

pred_original <- ifelse(prob_original > 0.5, 1, 0)


# Accuracy + Confusion Matrix
mean(pred_original == test_original$Outcome)
cm_original <- table(
  Predicted = pred_original,
  Actual = test_original$Outcome
)

cm_original
cat("\n")
AIC(model_original)
cat("\n")
BIC(model_original)

[1] 0.7922078

         Actual
Predicted  0  1
        0 92 22
        1 10 30

[1] 601.5482

[1] 641.3282

# PCA

In [ ]:
Xo_train <- train_original[, names(train_original) != "Outcome"]
Xo_test  <- test_original[, names(test_original) != "Outcome"]
yo_train <- train_original$Outcome
yo_test  <- test_original$Outcome

pca_model_original <- prcomp(Xo_train, center = TRUE, scale. = TRUE)

pc_train_original <- predict(pca_model_original, Xo_train)
pc_test_original  <- predict(pca_model_original, Xo_test)

pc_train_original <- data.frame(Outcome = yo_train, pc_train_original[, 1:5])
pc_test_original <- data.frame(Outcome = yo_test, pc_test_original[, 1:5])

model_original_pca <- glm(Outcome ~ ., data = pc_train_original, family = binomial)

In [ ]:
prob_original_pca <- predict(model_original_pca, newdata = pc_test_original, type = "response")
pred_original_pca <- ifelse(prob_original_pca > 0.5, 1, 0)

mean(pred_original_pca == pc_test_original$Outcome)
table(Predicted = pred_original_pca, Actual = pc_test_original$Outcome)

AIC(model_original_pca)
BIC(model_original_pca)

[1] 0.7662338

         Actual
Predicted  0  1
        0 90 24
        1 12 28

[1] 637.8278

[1] 664.3478

# K-means

In [ ]:
Xo_train <- train_original[, names(train_original) != "Outcome"]
Xo_test  <- test_original[, names(test_original) != "Outcome"]

Xo_train_scaled <- scale(Xo_train)
Xo_test_scaled <- scale(Xo_test, center = attr(Xo_train_scaled, "scaled:center"), scale  = attr(Xo_train_scaled, "scaled:scale"))

set.seed(123)
k2_original <- kmeans(Xo_train_scaled, centers = 2, nstart = 25)

train_original$Cluster <- as.factor(k2_original$cluster)
test_original$Cluster <- as.factor(apply(Xo_test_scaled, 1, function(x) {which.min(colSums((t(k2_original$centers) - x)^2))}))

model_original_kmeans <- glm(Outcome ~ ., data = train_original, family = binomial)

prob_original_kmeans <- predict(model_original_kmeans, newdata = test_original, type = "response")
pred_original_kmeans <- ifelse(prob_original_kmeans > 0.5, 1, 0)


mean(pred_original_kmeans == test_original$Outcome)
table(Predicted = pred_original_kmeans, Actual = test_original$Outcome)
AIC(model_original_kmeans)
BIC(model_original_kmeans)

[1] 0.7857143

         Actual
Predicted  0  1
        0 91 22
        1 11 30

[1] 603.3772

[1] 647.5772

# SVM

In [ ]:
install.packages("e1071")
library(e1071)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘proxy’




    Linear

In [ ]:
# Remove K-means cluster column if present
train_original$Cluster <- NULL
test_original$Cluster <- NULL

train_original$Outcome <- as.factor(train_original$Outcome)
test_original$Outcome <- as.factor(test_original$Outcome)

svm_original_linear <- svm(
  Outcome ~ .,
  data = train_original,
  kernel = "linear"
)

pred_original_linear <- predict(
  svm_original_linear,
  newdata = test_original
)

mean(pred_original_linear == test_original$Outcome)

table(
  Predicted = pred_original_linear,
  Actual = test_original$Outcome
)

[1] 0.7662338

         Actual
Predicted  0  1
        0 89 23
        1 13 29

In [ ]:
train_original$Outcome <- as.factor(train_original$Outcome)
test_original$Outcome <- as.factor(test_original$Outcome)

In [ ]:
svm_original_linear <- svm(
  Outcome ~ .,
  data = train_original,
  kernel = "linear"
)

pred_original_linear <- predict(
  svm_original_linear,
  newdata = test_original
)

mean(pred_original_linear == test_original$Outcome)
table(
  Predicted = pred_original_linear,
  Actual = test_original$Outcome
)

[1] 0.7662338

         Actual
Predicted  0  1
        0 89 23
        1 13 29

    Radial

In [ ]:
svm_original_radial <- svm(
  Outcome ~ .,
  data = train_original,
  kernel = "radial"
)

pred_original_radial <- predict(
  svm_original_radial,
  newdata = test_original
)

mean(pred_original_radial == test_original$Outcome)

table(
  Predicted = pred_original_radial,
  Actual = test_original$Outcome
)

[1] 0.7662338

         Actual
Predicted  0  1
        0 92 26
        1 10 26

# KNN

In [ ]:
install.packages("class")
library("class")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
Xo_train <- train_original[, !(names(train_original) %in% c("Outcome", "Cluster"))]
Xo_test  <- test_original[, !(names(test_original) %in% c("Outcome", "Cluster"))]
#Xo_train <- train_original[, names(train_original) != "Outcome"]
#Xo_test  <- test_original[, names(test_original) != "Outcome"]
yo_train <- train_original$Outcome
yo_test  <- test_original$Outcome

Xo_train_scaled <- scale(Xo_train)
Xo_test_scaled <- scale(Xo_test, center = attr(Xo_train_scaled, "scaled:center"), scale  = attr(Xo_train_scaled, "scaled:scale"))


k_values <- c(1,3,5,7,9,11,15)
acc_original <- sapply(k_values, function(k){pred <- knn(train = Xo_train_scaled, test = Xo_test_scaled, cl = yo_train, k = k)
  mean(pred == yo_test)
})

data.frame(k = k_values, accuracy = acc_original)

k,accuracy
<dbl>,<dbl>
1,0.7207792
3,0.7402597
5,0.7597403
7,0.7467532
9,0.7467532
11,0.7662338
15,0.7662338


In [ ]:
knn_original <- knn(train = Xo_train_scaled, test = Xo_test_scaled, cl = yo_train, k = 11)
mean(knn_original == yo_test)
table(Predicted = knn_original, Actual = yo_test)

[1] 0.7662338

         Actual
Predicted  0  1
        0 89 23
        1 13 29